# ImFusion CT Reconstruction

This notebook demonstrates cone-beam CT reconstruction using ImFusion CT. We simulate projections and reconstruct a volume using FDK (and the same API can run iterative solvers like MLEM, SART, CG, SQS).


## Overview
This tutorial reconstructs a CT volume from simulated projections using the FDK algorithm. The same API supports iterative solvers (MLEM, SART, CG, SQS) when you need higher-quality reconstructions at the expense of runtime.

### Objectives
- Simulate a projection stack from a given CT volume
- Reconstruct with FDK
- Visualize results

### Prerequisites
- A working `imfusion-sdk` and `imfusion-sdk-computed_tomography` installation with a valid license

## Setup

### Setup python path
This step can be skipped if the package got installed from PyPI.

In [1]:
# Setup Python path for ImFusion bindings
import sys
import os

# Add the build directory to Python path (adjust if needed)
build_lib_path = '/Users/wieczorek/Desktop/Dev/imfusionsuite/cmake-build-release/lib'
if build_lib_path not in sys.path:
    sys.path.insert(0, build_lib_path)

### Import the modules
We need to import `imfusion` and `imfusion.computed_tomography`.

In [2]:
try:
    import imfusion
    import imfusion.computed_tomography as ct
except ImportError as e:
    raise ImportError("Failed to import ImFusion CT bindings. Make sure the CT plugin is built and the path is correct.") from e

### Data setup
Unzips demo data if needed and available.

In [3]:
import os, sys
sdk_path = os.path.abspath(os.path.join(os.getcwd(), '../imfusion-sdk'))
if sdk_path not in sys.path:
    sys.path.insert(0, sdk_path)

try:
    from demo_utils import unzip_folder
    zip_path = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct.zip')
    data_dir = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct')
    if os.path.exists(zip_path) and not os.path.isdir(data_dir):
        print('Unzipping demo data...')
        unzip_folder(zip_path)
    else:
        print('Demo data is present or archive not found. Skipping unzip.')
except Exception as e:
    print(f"Data setup skipped ({e}).")


Demo data is present or archive not found. Skipping unzip.


## Steps in this notebook
1) Load a CT volume and simulate a half-scan trajectory
2) Reconstruct using FDK via `ct.Reconstruction`

### Load a CT volume and simulate a half-scan trajectory


In [4]:
import numpy as np
volume = imfusion.load("../imfusion-sdk/data/pet-ct-rtstruct/ct.imf")[0]
mat = volume.matrix()
mat[0:3, 3] = [0.0, 0.0, 0.0]
volume.set_matrix(mat)

simulator = ct.ConeBeamSimulation(
    volume[0],
    geometry_preset=ct.GeometryPreset.HALF_SCAN,
    proj_type=ct.ProjectionType.LOG_CONVERTED_ATTENUATION,
    width=1024,
    height=1024,
    frames=360,
    add_poisson_noise=False)
simulator.geometry().source_pat_distance = 900.0
simulator.geometry().det_size_x = 500.0
simulator.geometry().det_size_y = 500.0
projections = simulator()


### Reconstruct using FDK


In [5]:
reconstructor = ct.Reconstruction(
    projections,
    solver_mode="FDK",  # "MLEM", "SART", "CG", "SQS"
    subset_size=10,
)
reconstructed = reconstructor()

imfusion.show([projections, reconstructed])
